# Reading and concatenating numbered csv files

Some of our shared folders (e.g. screaming frog output) contain data split across many numbered csv files with the same columns, e.g. `html_1.csv`, `html_2.csv`, ... or `pdf_1.csv`, `pdf_2.csv`, ...

This notebook defines two helper functions:

1. `read_and_concat_csv_files` - reads any number of csv files from a shared s3 folder and concatenates them together.
2. `read_and_concat_to_from_columns` - does the same, but keeps only the `To` and `From` columns.

To point these at a different folder (e.g. html vs pdf), change the `path`, `prefix` and `n_files` variables in the usage cells below.

In [ ]:
import pandas as pd
import os
import audit_tool_streamlit.data_creation as dcr
import pandas
import psycopg2
import sqlalchemy

In [ ]:
### this is to create to files to run on screaming frog
# incremental updates in future should only need to run those that have changed (last updated date)
engine = sqlalchemy.create_engine('postgresql://', execution_options={"stream_results": True})
df = pandas.read_sql(sqlalchemy.text("""SELECT public_url FROM \"dbt\".\"gov_uk_content__regulation_notaxons\""""), engine)
output_dir = "chunks"
os.makedirs(output_dir, exist_ok=True)

chunk_size = 10_000

for i, start in enumerate(range(0, len(df), chunk_size), start=1):
    chunk = df.iloc[start:start + chunk_size]

    with open(f"{output_dir}/chunk_{i:02d}.txt", "w") as f:
        f.write("\n".join(chunk["public_url"].astype(str)))

print(f"Created {i} files")

## Function definitions

In [ ]:
def generate_file_names(prefix, n_files, extension="csv", start=1):
    """
    Builds a list of numbered file names, e.g. prefix="html", n_files=16 ->
    ["html_1.csv", "html_2.csv", ..., "html_16.csv"]
    params:
        prefix: string. e.g. "html" or "pdf"
        n_files: int. how many numbered files to generate
        extension: string. file extension, without the dot
        start: int. the number to start counting from
    returns:
        list(str)
    """
    return [f"{prefix}_{i}.{extension}" for i in range(start, start + n_files)]


def read_and_concat_csv_files(file_names, path, team="analysis_group_ds", **kwargs):
    """
    Reads any number of csv files from a shared s3 folder and concatenates them together.
    Assumes all files have the same columns.
    Change `path` to point at a different folder (e.g. the html folder vs the pdf folder).
    params:
        file_names: list(str). names of the csv files to read, e.g. from generate_file_names
        path: string. the s3 folder the files live in
        team: string. the s3 team schema to read from
        **kwargs: passed through to pd.read_csv (e.g. dtype, parse_dates)
    returns:
        pd.DataFrame
    """
    data_list = [
        dcr.read_csv_from_shared_folder(file_name=file_name, path=path, team=team, **kwargs)
        for file_name in file_names
    ]
    return pd.concat(data_list, ignore_index=True)


def read_and_concat_to_from_columns(file_names, path, team="analysis_group_ds", **kwargs):
    """
    Same as read_and_concat_csv_files, but keeps only the "To" and "From" columns.
    """
    data = read_and_concat_csv_files(file_names, path=path, team=team, **kwargs)
    return data[["To", "From"]]

## Usage

Change `path`, `prefix` and `n_files` to switch which folder/set of files you read from.

In [ ]:
# example: 16 html files, e.g. html_1.csv ... html_16.csv
html_path = "RLP/screaming_frog/html_internal/"
html_files = generate_file_names(prefix="html", n_files=16)

html_data = read_and_concat_csv_files(html_files, path=html_path)
html_to_from = read_and_concat_to_from_columns(html_files, path=html_path)

In [ ]:
# example: same thing, but for a different folder of numbered pdf files
pdf_path = "RLP/screaming_frog/pdf/"
pdf_files = generate_file_names(prefix="pdf", n_files=16)

pdf_data = read_and_concat_csv_files(pdf_files, path=pdf_path)
pdf_to_from = read_and_concat_to_from_columns(pdf_files, path=pdf_path)

In [ ]:
# you will need to do a join to main (old) data in tool and any matches need to keep the new data from screaming frog
# this then gets matched as previously to GA